In [4]:
import pandas as pd
from pathlib import Path
import os
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re # regular expressions
import odf

In [5]:
# Import cvs files nd place into dictinoary

print(os.getcwd())
print(os.listdir("."))

# place all data sources into folder call "data" alongside this .ipynb file
data_location = Path("data")


# Create a dictionary into which we wil aec the four dataframes and align them with the file names, for ease of access
# This will save us time cleaning the dfs individually with any repitive tasks 
data_sources= {}

print(data_location)

for i in sorted(data_location.glob("*.csv")):
    #strip back file name to just prefix, split at every dot and take first item and convert to lower case for consistency
    stripped_dfname= i.stem.split(".")[0].lower()
    # read csv file into a dataframe - red as utf8 and strip the hidden byte orser mark... as gettng corrupt characters..
    df = pd.read_csv(i, header=0, encoding="utf-8-sig")
    # standardise column names
    df.columns =df.columns.str.lower().str.replace(" ", "_")
    # stpre thb dataframe in the dictionary
    data_sources[stripped_dfname] = df
    # have a llok to make sure file names match into dataframe name in the dictionary and check shape to verify df is good
    print(f"{stripped_dfname} : {df.shape} <-{i.name}")


# Call the dictionary key names to show the dataframes present..
data_sources.keys()




e:\GG\Msc\SPEC9995_data_visualisation_marisa llorens-salvador_tues\github\spec9995_data_vis_marissa_tues\assignment 1\working
['working_dea_1.ipynb', 'steps_log.txt', 'data', 'data_overview.txt', 'eda_core_v1.png', 'eda_net_debt_waterfall_v1.png', 'core_features.csv', 'eda_core_v2.png', 'eda_core_v3.png', 'working_dea_v2.ipynb', 'tableau_all_long.csv', 'tableau_core_long.csv', 'tableau_core_wide.csv', 'working_v3.ipynb', '~WRL1103.tmp', 'assignment1_cso_hvd_eamonn_kelly_D24127620.pdf.docx', 'exported', 'images', 'assignment1_cso_hvd_eamonn_kelly_D24127620)_v2.pdf.docx', 'CSO -  Revenue vs Expenditures.twb', 'CSO -  Surplus vs Deficit.twb', 'CSO -  Gross and Net debt with Cash & Bond Assets.twb', 'CSO -  Tax Composition - Corp_Tax_Income_Tax_VAT.twb', 'working_v4.ipynb', 'working_v5.ipynb', 'assignment1_cso_hvd_eamonn_kelly_D24127620)_v3.pdf.docx', '~$signment1_cso_hvd_eamonn_kelly_D24127620)_v4.pdf.docx', '~$signment1_cso_hvd_eamonn_kelly_D24127620)_v3.pdf.docx', 'CSO - Revenue Growth 

dict_keys(['gfa01', 'gfa02', 'gfa03', 'gfa04', 'gfq02'])

In [6]:
# Creating a dictionary definition of what to keep from the downloaded csv files, mapping the retained data to new feature names in dictionary key value pairs
keep = {
    "gfa01": {
        "General Government transactions - Revenue - ESA2010 Code (TR)": "total_revenue",
        "General Government transactions - Expenditure - ESA2010 Code (TE)": "total_expenditure",
        "General Government Surplus/Deficit - ESA2010 Code (B9)": "surplus_deficit",
        "General Government transactions - Revenue - Capital transfer revenue (excluding taxes) - ESA2010 Code (D9N)": "one_off_capital_receipt",
        "General Government transactions - Expenditure - Interest (excluding FISIM) - ESA2010 Code (D41)": "interest_debt",
    },
    "gfa02": {
        "Financial net worth at market value, excluding pension liabilities - EDP debt instruments (Asset) - ESA2010 Code (F2 + F3 + F4)": "cash_bond_assets",
        "Gross General Government Debt (EDP face value) - Gross General Government Debt - ESA2010 Code (GGDebt)": "gross_debt",
        "Gross General Government Debt (EDP face value) - Net General Government Debt - ESA2010 Code (GGNetDebt)": "net_debt",
    },
    "gfa03": {
        "Revenue - Total Taxes - ESA2010 Code (D2 + D5 + D91)": "total_tax",
        "Revenue - Corporation tax - ESA2010 Code (D51 part of)": "ct",
        "Revenue - Income tax - ESA2010 Code (D51 part of)": "income_tax",
        "Revenue - Taxes on products of which Value-Added Tax - ESA2010 Code (D211)": "vat",
    },
    "gfa04": {
        "Expenditure - Gross fixed capital formation - ESA2010 Code (P51)": "public_investment",
        "Expenditure - Total Social benefits - ESA2010 Code (D62 + D632)": "social_benefits",
        "Expenditure - Other capital transfers - ESA2010 Code (D99)": "capital_transfers_paid",
    },
}

In [7]:
# filter and rename the features fromt he dictioary that we want to keep

# create a list to store the selected data series, each table resutls get stacked in this, at the end of each other, sequentially
stacked_gfadata_list = []
for table, mapping in keep.items():
    # we'll sue a working copy dictionary ut 
    df = data_sources[table].copy()
    # use regex to strip any double spaces, >=1 space replaced with 1 space, and remove leding an trailing
    df["item"] = df["item"].str.replace(r"\s+", " ", regex=True).str.strip()  
    # deal with any missing values in our mapping, using sets to remove items nto in the keep dict we defined earlier
    missing = set(mapping) - set(df["item"])       
    if missing:
        print(f"WARNING {table}: not found -> {missing}")

    # keep only the rows in that mapping
    df = df[df["item"].isin(mapping)].copy()
    # map each rows label and gets short name from dict item thats put in df series
    df["feature_name"] = df["item"].map(mapping)
    # create new label names for the df and put in the values
    df["orig_org_source"], df["orig_file_source"], df["is_forecast"] = "CSO", table.upper(), False
    # append to the defined list
    stacked_gfadata_list.append(df)
    # pritn stament - troubleshooting...check what we have . print the mapping
    # table is the name in the dictionary for the csv filename abbrev used in the dictionary
    print(f"{table} : {df.shape[0]} rows <- {len(mapping)} features over a period of {df['year'].nunique()} years "
          f"({df['year'].min()} to {df['year'].max()})")

# put it all table into new dataframe called df_cso
df_cso = pd.concat(stacked_gfadata_list, ignore_index=True)[
    ["year", "feature_name", "value", "unit", "orig_org_source", "orig_file_source", "is_forecast"]]

# print the totals - the shape of the dataframe
print("\ndf_cso:", df_cso.shape, "| total number of features", df_cso["feature_name"].nunique())
display(df_cso.head())

gfa01 : 155 rows <- 5 features over a period of 31 years (1995 to 2025)
gfa02 : 78 rows <- 3 features over a period of 26 years (2000 to 2025)
gfa03 : 104 rows <- 4 features over a period of 26 years (2000 to 2025)
gfa04 : 78 rows <- 3 features over a period of 26 years (2000 to 2025)

df_cso: (415, 7) | total number of features 15


,year,feature_name,value,unit,orig_org_source,orig_file_source,is_forecast
0,1995,total_revenue,20959.0,Euro Million,CSO,GFA01,False
1,1995,one_off_capital_receipt,911.0,Euro Million,CSO,GFA01,False
2,1995,total_expenditure,22096.0,Euro Million,CSO,GFA01,False
3,1995,interest_debt,2814.0,Euro Million,CSO,GFA01,False
4,1995,surplus_deficit,-1137.0,Euro Million,CSO,GFA01,False


In [8]:
print(df_cso.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 415 entries, 0 to 414
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              415 non-null    int64  
 1   feature_name      415 non-null    object 
 2   value             415 non-null    float64
 3   unit              415 non-null    object 
 4   orig_org_source   415 non-null    object 
 5   orig_file_source  415 non-null    object 
 6   is_forecast       415 non-null    bool   
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 20.0+ KB
None


In [9]:
# EXploratory Data Analysis

print(df_cso.describe())

              year          value
count   415.000000     415.000000
mean   2011.566265   37157.248193
std       8.169687   52265.294915
min    1995.000000  -53764.000000
25%    2005.000000    3887.000000
50%    2012.000000   15514.000000
75%    2019.000000   44555.000000
max    2025.000000  235182.000000


In [10]:
# quick look at data to see if its clean, no duplicates, null values unusual characters etc
# convert it to d first so can easily run some checks over it
# df=pd.DataFrame(processed_per_file_data)
# view various dataframe details
# checking for null values and specific unknown values
print ("Rows     : " , df_cso.shape[0])
print ("Columns  : " , df_cso.shape[1])
print('------')
print ("\nFeatures : \n" , df_cso.columns.tolist())
print('------')
print ("\nMissing values :  ", df_cso.isnull().sum().values.sum())
print('------')
print ("\nMissing values per column:  ", df_cso.isnull().sum())
print('------')
print('\nnum of dups in df   =   {}'.format(df_cso.duplicated().sum()))
print('------')
print ("\nUnique values :  \n", df_cso.nunique())
print('------')
print('\n null values = ', df_cso.isnull().values.any())
print('------')
df_cso_missing = (df_cso=='?').sum()
print('\"  ?   \" values present =  \n',df_cso_missing)
print('------')
df_cso_unknown = (df_cso=='unknown').sum()
print('\"   unknown   \" values present =  \n',df_cso_unknown)
print('------')
df_cso_na = df_cso.isin(['N/A', 'N\\A', 'NA', 'n/a', 'n\\A', 'na', 'N_A']).sum()
print('    NA     values present = \n', df_cso_na)
print('------')
df_cso_none = df_cso.isin(['None', 'none']).sum()
print('    none    values present = \n', df_cso_none)
print('------')
# regex https://stackoverflow.com/questions/2124010/grep-regex-to-match-non-ascii-characters
# count of rows per column containing non-ascii characters
rogue_ascii_df_cso = df_cso.apply(lambda s: s.astype(str).str.contains(r"[^\x00-\x7F]", regex=True, na=False).sum())
print("    non ascii chars   values present:\n", rogue_ascii_df_cso)

Rows     :  415
Columns  :  7
------

Features : 
 ['year', 'feature_name', 'value', 'unit', 'orig_org_source', 'orig_file_source', 'is_forecast']
------

Missing values :   0
------

Missing values per column:   year                0
feature_name        0
value               0
unit                0
orig_org_source     0
orig_file_source    0
is_forecast         0
dtype: int64
------

num of dups in df   =   0
------

Unique values :  
 year                 31
feature_name         15
value               413
unit                  1
orig_org_source       1
orig_file_source      4
is_forecast           1
dtype: int64
------

 null values =  False
------
"  ?   " values present =  
 year                0
feature_name        0
value               0
unit                0
orig_org_source     0
orig_file_source    0
is_forecast         0
dtype: int64
------
"   unknown   " values present =  
 year                0
feature_name        0
value               0
unit                0
orig_org_sourc

In [11]:
# checkl the -71,556 value in values.... can ahve negative values
display(df_cso.loc[df_cso["value"].idxmin()])
display(df_cso[df_cso["value"] < 0].groupby("feature_name")["value"].agg(["min", "count"]))

year                           2010
feature_name        surplus_deficit
value                      -53764.0
unit                   Euro Million
orig_org_source                 CSO
orig_file_source              GFA01
is_forecast                   False
Name: 79, dtype: object

,min,count
feature_name,,
surplus_deficit,-53764.0,15


In [12]:
# craeting calculated features from CSO data
# NORM_START, NORM_END = 2000, 2014        # pre-windfall period for the "normal" CT share

# finction which will take 4 inputs i.e. df, new feature name, calcualted values for new feature (one per year), and unit value
def add_feature(df, name, series, unit, org="CSO"):
    #Remove any existing rows for `name`, then append `series` (indexed by year) in df_cso format
    # keep all rows except  the new data row  "feature name" if it exists - allows to rerun, so if "feature aname " alreadyh exosts remove it
    df = df[df["feature_name"] != name]               
    # remove any blank values in the row  , some data i missing for 1995 to 1999, in some cases. tax data start in 2000
    s = series.dropna()
    # we'll defnie what goes into the existing columns where blanks will appear for the new feature
    rows = pd.DataFrame({
        "year": s.index,
        "feature_name": name,
        "value": s.values,
        "unit": unit,
        "orig_org_source": org,
        "orig_file_source": "calculated", # label it as claculated
        "is_forecast": False,
    })
    return pd.concat([df, rows], ignore_index=True)

In [13]:
# crate other tax value - subtract Corp Tax + Income tax + VAY tax from total tax to get toehr tax amounts...
# we pivot the dataframe by year to get year per row and the features per column, into a Wide view 
# # this wll make it easier and give us one column per feature, one row per year, for calculating across features
# we keep thsi separate from the dataframe we add to , we calcualet off this and then add results to dataframe
df_cso_wide = df_cso.pivot(index="year", columns="feature_name", values="value")


In [14]:
# Add "other_tax" calculated feature to dataframe
# ceate  working dataframe off the existing one, to allow for rollback whilr troubleshooting
df_cso_1 = df_cso
df_cso_1 = add_feature(df_cso_1, "other_tax", df_cso_wide["total_tax"] - df_cso_wide["ct"] - df_cso_wide["income_tax"] - df_cso_wide["vat"], "Euro Million")




In [15]:
df_cso_1.shape
# print the totals - the shape of the dataframe
print("\ndf_cso_1:", df_cso_1.shape, "| total number of features", df_cso_1["feature_name"].nunique())


df_cso_1: (441, 7) | total number of features 16


In [16]:
# Add Coroporation tax percentage of total tax value i.e. its share of total tax
# thi is a value we wil want to plot later on, so calculating and adding now

# creat working dataframe off existing one
df_cso_2 = df_cso_1

df_cso_2 = add_feature(df_cso_2, "ct_pct_tot_tax", df_cso_wide["ct"] / df_cso_wide["total_tax"] * 100, "Percent")

In [17]:
print("\ndf_cso_2:", df_cso_2.shape, "| total number of features", df_cso_2["feature_name"].nunique())


df_cso_2: (467, 7) | total number of features 17


In [18]:
   # calculate the average value for 
   # NORM_START, NORM_END = 2000, 2014        # pre-windfall period for the "normal" CT share
   
   # df_cso_wide_2 = df_cso_2.pivot(index="year", columns="feature_name", values="value")
   # print("other_tax check:", (df_cso_wide_2["ct"] + df_cso_wide_2["income_tax"] + df_cso_wide_2["vat"] + df_cso_wide_2["other_tax"] - df_cso_wide_2["total_tax"]).abs().max())
   # print(df_cso_wide_2["ct_pct_tot_tax"].loc[[2000, 2014, 2025]].round(1))
   # print("normal CT share 2000-2014:", round(w2["ct_pct_tot_tax"].loc[2000:2014].mean(), 2), "%")

In [19]:
# Create calculated feature "ct_diff_from_xpctd "  corporation tax value difference from expected CT value i.e. the excess CT
# we'll sue the wide version of the data frame crated earlier for ease of calculation then ppulate dataframe wiht the calculations
# to calculate the normal corporation tax we take a period of normal CT and get the average ver that persio
# thsi comes to approx 11.86% as outlined in previous cell

# creating workign wide and normal dataframe
df_cso_wide_3 =  df_cso_2.pivot(index="year", columns="feature_name", values="value")
df_cso_3 = df_cso_2

# pre-windfall time basedon the oeriod 2000 to 2014
norm_start = 2000
norm_end = 2014 

norm = df_cso_wide_3["ct_pct_tot_tax"].loc[norm_start:norm_end].mean() /100

# have a lok at the valoues for 2000, 2014 and 2025
print(df_cso_wide_3["ct_pct_tot_tax"].loc[[2000, 2014, 2025]].round(1))
# get the average value over the 2000 to 2014 period
print("normal CT share 2000-2014:", round(df_cso_wide_3["ct_pct_tot_tax"].loc[2000:2014].mean(), 2), "%")


# creatign working datframes to alow rol back
# df_cso_wide_3 =  df_cso_wide_2 
# df_cso_3 = df_cso_2
# norm = (df_cso_wide["ct"] / df_cso_wide["total_tax"]).loc[NORM_START:NORM_END].mean()

ct_diff_from_xpctd = df_cso_wide_3["ct"] - norm * df_cso_wide_3["total_tax"]
df_cso_3 = add_feature(df_cso_2, "ct_diff_from_xpctd", ct_diff_from_xpctd, "Euro Million")

print(ct_diff_from_xpctd.loc[[2002, 2011, 2025]].round(0))



year
2000    13.3
2014     9.9
2025    29.5
Name: ct_pct_tot_tax, dtype: float64
normal CT share 2000-2014: 11.86 %
year
2002      922.0
2011    -1110.0
2025    19681.0
dtype: float64


In [20]:
print("\ndf_cso_3:", df_cso_3.shape, "| total number of features", df_cso_3["feature_name"].nunique())


df_cso_3: (493, 7) | total number of features 18


In [21]:
# create workign dfs
# need to sue calculation from previos one as se din this so making new df_cso_3 as working wide df to use for calculations
df_cso_wide_4 =  df_cso_wide_3
df_cso_4 = df_cso_3

# craete budget surplus minus the excess corporation tax find value if over or under amount of revenu received
# norm = (df_cso_3["ct"] / df_cso_3["total_tax"]).loc[NORM_START:NORM_END].mean()

bdgt_srplus_minus_ct_diff_from_xpctd = df_cso_wide_4["surplus_deficit"] - ct_diff_from_xpctd.clip(lower=0) 
df_cso_4 = add_feature(df_cso_3, "bdgt_srplus_minus_ct_diff_from_xpctd", bdgt_srplus_minus_ct_diff_from_xpctd, "Euro Million")

print(bdgt_srplus_minus_ct_diff_from_xpctd.loc[[2011, 2022, 2025]].round(0))





year
2011   -23320.0
2022    -3462.0
2025    -9014.0
dtype: float64


In [22]:
print("\ndf_cso_4:", df_cso_4.shape, "| total number of features", df_cso_4["feature_name"].nunique())


df_cso_4: (519, 7) | total number of features 19


In [23]:
df_cso_final = df_cso_4
print(df_cso_final.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 519 entries, 0 to 518
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              519 non-null    int64  
 1   feature_name      519 non-null    object 
 2   value             519 non-null    float64
 3   unit              519 non-null    object 
 4   orig_org_source   519 non-null    object 
 5   orig_file_source  519 non-null    object 
 6   is_forecast       519 non-null    bool   
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 25.0+ KB
None


In [24]:

print(df_cso_final.describe())

              year          value
count   519.000000     519.000000
mean   2011.753372   30164.578908
std       8.048409   48989.349018
min    1995.000000  -53764.000000
25%    2005.000000    1432.005469
50%    2012.000000   11527.000000
75%    2019.000000   36935.000000
max    2025.000000  235182.000000


In [25]:
# quick look at data to see if its clean, no duplicates, null values unusual characters etc
# convert it to d first so can easily run some checks over it
# df=pd.DataFrame(processed_per_file_data)
# view various dataframe details
# checking for null values and specific unknown values
print ("Rows     : " , df_cso_final.shape[0])
print ("Columns  : " , df_cso_final.shape[1])
print('------')
print ("\nFeatures : \n" , df_cso_final.columns.tolist())
print('------')
print ("\nMissing values :  ", df_cso_final.isnull().sum().values.sum())
print('------')
print ("\nMissing values per column:  ", df_cso_final.isnull().sum())
print('------')
print('\nnum of dups in df   =   {}'.format(df_cso_final.duplicated().sum()))
print('------')
print ("\nUnique values :  \n", df_cso_final.nunique())
print('------')
print('\n null values = ', df_cso_final.isnull().values.any())
print('------')
df_cso_missing = (df_cso_final=='?').sum()
print('\"  ?   \" values present =  \n',df_cso_missing)
print('------')
df_cso_unknown = (df_cso_final=='unknown').sum()
print('\"   unknown   \" values present =  \n',df_cso_unknown)
print('------')
df_cso_na = df_cso_final.isin(['N/A', 'N\\A', 'NA', 'n/a', 'n\\A', 'na', 'N_A']).sum()
print('    NA     values present = \n', df_cso_na)
print('------')
df_cso_none = df_cso_final.isin(['None', 'none']).sum()
print('    none    values present = \n', df_cso_none)
print('------')
# regex https://stackoverflow.com/questions/2124010/grep-regex-to-match-non-ascii-characters
# count of rows per column containing non-ascii characters
rogue_ascii_df_cso = df_cso_final.apply(lambda s: s.astype(str).str.contains(r"[^\x00-\x7F]", regex=True, na=False).sum())
print("    non ascii chars   values present:\n", rogue_ascii_df_cso)

Rows     :  519
Columns  :  7
------

Features : 
 ['year', 'feature_name', 'value', 'unit', 'orig_org_source', 'orig_file_source', 'is_forecast']
------

Missing values :   0
------

Missing values per column:   year                0
feature_name        0
value               0
unit                0
orig_org_source     0
orig_file_source    0
is_forecast         0
dtype: int64
------

num of dups in df   =   0
------

Unique values :  
 year                 31
feature_name         19
value               510
unit                  2
orig_org_source       1
orig_file_source      5
is_forecast           1
dtype: int64
------

 null values =  False
------
"  ?   " values present =  
 year                0
feature_name        0
value               0
unit                0
orig_org_source     0
orig_file_source    0
is_forecast         0
dtype: int64
------
"   unknown   " values present =  
 year                0
feature_name        0
value               0
unit                0
orig_org_sourc

In [26]:
# export to csv so cna load into Tableau and do explorattory data visualisations
# define the path and create the folder if it doesn' exist
export_location = Path("exported")
export_filename_cso = ("df_cso_v2.csv")
export_location.mkdir(exist_ok=True)

df_cso_final.to_csv(export_location/export_filename_cso , index=False)
print(f" saved file : {(export_location/export_filename_cso).resolve()}")

 saved file : E:\GG\Msc\SPEC9995_data_visualisation_marisa llorens-salvador_tues\github\spec9995_data_vis_marissa_tues\assignment 1\working\exported\df_cso_v2.csv


In [27]:
check = pd.read_csv(export_location / export_filename_cso)
# check the shape
print(f" shape {check.shape} - ( == expecting 519 rows and 7 features == )")
# checlk the number of features types
print(f" features {check["feature_name"].nunique()} ( ==  expecting 19 features types == ) ")

 shape (519, 7) - ( == expecting 519 rows and 7 features == )
 features 19 ( ==  expecting 19 features types == ) 


# IFAC Data

In [28]:
# # Find IFAC data packs in the data folder by the year in the file name 
# ifac_xlsx_files = {}
# for f in sorted(data_location.glob("*.xlsx")):
#     # ignroe temp files with ~$ if any exist
#     if f.name.startswith("~$"):
#         continue
#     year = re.search(r"20\d{2}", f.name)
#     if year:
#         ifac_xlsx_files[int(year.group())] = f
#         print(f"{year.group()} : {f.name}")

#     # # have a llok to make sure file names match into dataframe name in the dictionary and check shape to verify df is good
#     # print(f"{stripped_dfname} : {df.shape} <-{i.name}")


# xl_ifac = ifac_xlsx_files[2026]

In [29]:
# Location of each series in the IFAC June 2026 data pack, Excel row/column numbers....
# create nested dictionary with contains the manual location of the data we want to put into dataframe
# manual process to get the data from specific tabs
# we use the tab name, the first number if the the year row, then rows containing the data numbers, use a list value as multiple numbers, include unit text, single digit is multipler to get all value sin same scale i.e. millions or billions, then label valeus as per the tabs themselves, GG stands for general government as per xlsx
# key: (tab name, year row, first data column, unit, scale, {data row: feature_name})
# scale converts units where needed  i.e. Figure 2.10 is in EUR billion -> x1000 = EUR million
ifac2026_tabs_dict={
    #surplus exclusing apple receipt , inlcuding forecasted avlues
    "2.1_srpls_excl_apl":  ("Figure 2.1", 33, 2, "Euro Million", 1, {34: "ifac_gg_surplus_excl_apple"}),
    # balans excluding excess CT and apple receipt
    "2.2_bal_excl_xcs_ct":  ("Figure 2.2", 41, 2, "Euro Million", 1, {44: "ifac_gg_bal_excl_excess_ct"}),
    # anual contribution into state funds and the surplus available to fudn them
    "2.9_state_fund_contribs":  ("Figure 2.9", 25, 4, "Euro Million", 1,
                {26: "ifac_ntf_contribution",
                 27: "ifac_sif_contribution",
                 28: "ifac_icnf_contribution",
                 29: "ifac_fif_contribution",
                 30: "ifac_gg_surplus_incl_apple"}),
    # balances held in the funds - it comes in billions, so x 1000 to ge millions
    "2.10_state_fund_bals": ("Figure 2.10", 23, 2, "Euro Million", 1000,
                {24: "ifac_sif_balance",
                 25: "ifac_fif_balance",
                 26: "ifac_ntf_balance",
                 27: "ifac_icnf_balance",
                 28: "ifac_isif_balance"}),
    # ct as% of national income in irl and EU average
    "3.4_ct_pct_ntnl_income":  ("Figure 3.4", 52, 2, "Percent", 1,
                {56: "ifac_ct_pct_natl_income_ie",
                 65: "ifac_ct_pct_natl_income_eu"}),
    # surplus as % ct i.e. the amount of ct saved....
    "3.5_pct_ct_saved":  ("Figure 3.5", 40, 3, "Percent", 1,
                {41: "ifac_pct_ct_saved"}),
}

# tab positions by data pack year. will manually map tab positions to year per data pack, 2023, 202 etc layouts are different per datapack and xls
# if mapping is not defined it will eb skipped
ifac_tabs_by_year = {2026: ifac2026_tabs_dict}

In [30]:
# Function to extract rows  and data from an IFAC tab, the years run across a row so different to csv data, then converts into into long format, witht he years runnign downwards and with the same 7 columns as df_cso
# after 2025 the values are forecasts, so we'll flag that
# xl file path(found using .glob), id file poer year in name, dict key, sheet, 
def ifac_block(xl, data_pack_year, tab_key, sheet, year_row, first_col, unit, scale, rows, last_actual=2025):
    # get the whoel sheet as a grid, wo[n't take header to stop row  1 being used as colimn names
    raw = pd.read_excel(xl, sheet_name=sheet, header=None)
    # get data value years from row in the sheet ege row 23 and blanks when no data, convert to numbers and turn blanks and text into nan of crashing out. the file name year comes from data_pack_year fed into function
    years = pd.to_numeric(raw.iloc[year_row - 1, first_col - 1:], errors="coerce")
    # carete a list to store results
    out = []
    # loop through the chose rows in the individual sheets, i.e. one sheet per fucntion call
    for r, feature in rows.items():
        # get the rows alues and conver units
        vals = pd.to_numeric(raw.iloc[r - 1, first_col - 1:], errors="coerce") * scale
        # pair each value with its year, drop empty columns past the data
        df = pd.DataFrame({"year": years.values, "value": vals.values}).dropna()
        # cponert years to whole numbers and add the descriptive rows
        df["year"] = df["year"].astype(int)
        df["feature_name"] = feature
        df["unit"] = unit
        # will label orig_source as IFAC
        df["orig_org_source"] = "IFAC"
        df["orig_file_source"] = f"far{data_pack_year}_{tab_key}"
        # after 2025 its a forecast so is_forecast = TRUE, not False as has been up to then
        df["is_forecast"] = df["year"] > last_actual  
        out.append(df)
    # stack all the rows and match column order to mach csv dataframe
    return pd.concat(out, ignore_index=True)[
        ["year", "feature_name", "value", "unit", "orig_org_source", "orig_file_source", "is_forecast"]]

In [31]:
# Loop through the tabs to extract the data into datafram in created dictionary
# the folder location where xlsx files are lcoated and to seacrch, print it for trobelshooting
print(data_location)

# loop xlsx files in data location
# owe will get one dataframe per tab, include Fiscal-Assessment to only inlcude IFAC files, exclude norway, UK and dutch files.
for i in sorted(data_location.glob("*Fiscal-Assessment*.xlsx")):
    # skip any temp files present identified with ~$
    if i.name.startswith("~$"):
        continue
    # get the data pack year from the file name using regex i.e. "...June-2026.xlsx" -> 2026
    # the files come wiht s data pack
    found = re.search(r"20\d{2}", i.name)
    # if can'[t find year in file name continue
    if not found:
        print(f"skipped (no year in name): {i.name}")
        continue
    # turn year into an int, this wil eb sued to match the key values inifac_tabs_by _year. found.group is te match eyar fro earlier
    pack_year = int(found.group())
    # only extract packs whose tab positions have been defined
    if pack_year not in ifac_tabs_by_year:
        print(f"skipped (no tab positions defined for {pack_year}): {i.name}")
        continue
    # extract each tab into its own dataframe and store in the earlier created dictionary
    for tab_key, (sheet, year_row, first_col, unit, scale, rows) in ifac_tabs_by_year[pack_year].items():
        # cal the function to extract the tab data
        df = ifac_block(i, pack_year, tab_key, sheet, year_row, first_col, unit, scale, rows)
        # name the source to populated orign_file_source values alongside gfa01, gfa02 etc
        key = f"ifac{pack_year}_{tab_key}"
        data_sources[key] = df
        print(f"{key} : {df.shape} <-{i.name} [{sheet}]")

# Call the dictionary key names to show the dataframes present
data_sources.keys()

data
ifac2026_2.1_srpls_excl_apl : (9, 7) <-Data-pack-Fiscal-Assessment-Report-June-2026.xlsx [Figure 2.1]
ifac2026_2.2_bal_excl_xcs_ct : (36, 7) <-Data-pack-Fiscal-Assessment-Report-June-2026.xlsx [Figure 2.2]
ifac2026_2.9_state_fund_contribs : (35, 7) <-Data-pack-Fiscal-Assessment-Report-June-2026.xlsx [Figure 2.9]
ifac2026_2.10_state_fund_bals : (35, 7) <-Data-pack-Fiscal-Assessment-Report-June-2026.xlsx [Figure 2.10]
ifac2026_3.4_ct_pct_ntnl_income : (61, 7) <-Data-pack-Fiscal-Assessment-Report-June-2026.xlsx [Figure 3.4]
ifac2026_3.5_pct_ct_saved : (9, 7) <-Data-pack-Fiscal-Assessment-Report-June-2026.xlsx [Figure 3.5]


dict_keys(['gfa01', 'gfa02', 'gfa03', 'gfa04', 'gfq02', 'ifac2026_2.1_srpls_excl_apl', 'ifac2026_2.2_bal_excl_xcs_ct', 'ifac2026_2.9_state_fund_contribs', 'ifac2026_2.10_state_fund_bals', 'ifac2026_3.4_ct_pct_ntnl_income', 'ifac2026_3.5_pct_ct_saved'])

In [32]:
# Stack all ifac dataframes into one df, in the same format as df_cso
# k is the dictionary "key" value,  and "v" is the dataframe value
# keep items string with ifac and list comprehenions to build a list of the 6 created dataframes in the dictionary
# stack them on top of each other using concat.. don't use dataframe indexes for numbering
df_ifac = pd.concat([v for k, v in data_sources.items() if k.startswith("ifac")], ignore_index=True)
print("df_ifac:", df_ifac.shape, "| total number of features", df_ifac["feature_name"].nunique())
display(df_ifac.head())

df_ifac: (185, 7) | total number of features 15


,year,feature_name,value,unit,orig_org_source,orig_file_source,is_forecast
0,2022,ifac_gg_surplus_excl_apple,8482.0,Euro Million,IFAC,far2026_2.1_srpls_excl_apl,False
1,2023,ifac_gg_surplus_excl_apple,7430.0,Euro Million,IFAC,far2026_2.1_srpls_excl_apl,False
2,2024,ifac_gg_surplus_excl_apple,9075.0,Euro Million,IFAC,far2026_2.1_srpls_excl_apl,False
3,2025,ifac_gg_surplus_excl_apple,11222.0,Euro Million,IFAC,far2026_2.1_srpls_excl_apl,False
4,2026,ifac_gg_surplus_excl_apple,8700.0,Euro Million,IFAC,far2026_2.1_srpls_excl_apl,True


In [33]:
print(df_ifac.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              185 non-null    int64  
 1   feature_name      185 non-null    object 
 2   value             185 non-null    float64
 3   unit              185 non-null    object 
 4   orig_org_source   185 non-null    object 
 5   orig_file_source  185 non-null    object 
 6   is_forecast       185 non-null    bool   
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 9.0+ KB
None


In [34]:
print(df_ifac.describe())

              year         value
count   185.000000    185.000000
mean   2018.394595   2810.006831
std      10.694928  11283.415538
min    1995.000000 -53764.000000
25%    2010.000000      2.950000
50%    2024.000000     31.165776
75%    2027.000000   5345.000000
max    2030.000000  51854.030000


In [35]:
# quick look at data to see if its clean, no duplicates, null values unusual characters etc
# convert it to d first so can easily run some checks over it
# df=pd.DataFrame(processed_per_file_data)
# view various dataframe details
# checking for null values and specific unknown values
print ("Rows     : " , df_ifac.shape[0])
print ("Columns  : " , df_ifac.shape[1])
print('------')
print ("\nFeatures : \n" , df_ifac.columns.tolist())
print('------')
print ("\nMissing values :  ", df_ifac.isnull().sum().values.sum())
print('------')
print ("\nMissing values per column:  ", df_ifac.isnull().sum())
print('------')
print('\nnum of dups in df   =   {}'.format(df_ifac.duplicated().sum()))
print('------')
print ("\nUnique values :  \n", df_ifac.nunique())
print('------')
print('\n null values = ', df_ifac.isnull().values.any())
print('------')
df_cso_missing = (df_ifac=='?').sum()
print('\"  ?   \" values present =  \n',df_cso_missing)
print('------')
df_cso_unknown = (df_ifac=='unknown').sum()
print('\"   unknown   \" values present =  \n',df_cso_unknown)
print('------')
df_cso_na = df_ifac.isin(['N/A', 'N\\A', 'NA', 'n/a', 'n\\A', 'na', 'N_A']).sum()
print('    NA     values present = \n', df_cso_na)
print('------')
df_cso_none = df_ifac.isin(['None', 'none']).sum()
print('    none    values present = \n', df_cso_none)
print('------')
# regex https://stackoverflow.com/questions/2124010/grep-regex-to-match-non-ascii-characters
# count of rows per column containing non-ascii characters
rogue_ascii_df_cso = df_ifac.apply(lambda s: s.astype(str).str.contains(r"[^\x00-\x7F]", regex=True, na=False).sum())
print("    non ascii chars   values present:\n", rogue_ascii_df_cso)

Rows     :  185
Columns  :  7
------

Features : 
 ['year', 'feature_name', 'value', 'unit', 'orig_org_source', 'orig_file_source', 'is_forecast']
------

Missing values :   0
------

Missing values per column:   year                0
feature_name        0
value               0
unit                0
orig_org_source     0
orig_file_source    0
is_forecast         0
dtype: int64
------

num of dups in df   =   0
------

Unique values :  
 year                 36
feature_name         15
value               166
unit                  2
orig_org_source       1
orig_file_source      6
is_forecast           2
dtype: int64
------

 null values =  False
------
"  ?   " values present =  
 year                0
feature_name        0
value               0
unit                0
orig_org_source     0
orig_file_source    0
is_forecast         0
dtype: int64
------
"   unknown   " values present =  
 year                0
feature_name        0
value               0
unit                0
orig_org_sourc

In [36]:
# we'll carete a final versin of ifac to keepm clean checkpoint version here to tool bak if we need
df_ifac_final = df_ifac

# well do some checks now make sure no potential conflicts joining
# same columns, same order, in both
print("columns match:", list(df_cso_final.columns) == list(df_ifac_final.columns))

# no feature name used by both sources (IFAC names are prefixed ifac_)
overlap = set(df_cso_final["feature_name"]) & set(df_ifac_final["feature_name"])
print("feature names occuring in both:", overlap if overlap else "none")

# # no duplicate feature/year rows
# print("duplicate year or feature rows:", df_cso_ifac.duplicated(subset=["year", "feature_name"]).sum())


columns match: True
feature names occuring in both: none


In [37]:
# combine CSO and IFAC dfs

df_cso_ifac = pd.concat([df_cso_final, df_ifac_final], ignore_index=True)


In [38]:
print(f" df_cso_final:  {df_cso_final.shape}")
print(f" df_ifac_final: {df_ifac_final.shape}")
print(f"=================================")
print(f" Combined  df_cso_ifac shape = {df_cso_ifac.shape} - expecting 704 rows (cso=519 + ifac =185) and 7 features (both =7)")

 df_cso_final:  (519, 7)
 df_ifac_final: (185, 7)
 Combined  df_cso_ifac shape = (704, 7) - expecting 704 rows (cso=519 + ifac =185) and 7 features (both =7)


In [39]:
# Export into csv we cna import import into Tableau then

# export to csv so cna load into Tableau and do explorattory data visualisations
# define the path and create the folder if it doesn' exist
print(f" export location = {export_location}" )
export_filename_cso_ifac = ("df_cso_ifac.csv")
export_location.mkdir(exist_ok=True)

df_cso_ifac.to_csv(export_location/export_filename_cso_ifac , index=False)
print(f" saved file : {(export_location/export_filename_cso_ifac).resolve()}")





 export location = exported
 saved file : E:\GG\Msc\SPEC9995_data_visualisation_marisa llorens-salvador_tues\github\spec9995_data_vis_marissa_tues\assignment 1\working\exported\df_cso_ifac.csv


# Norway Oil - Equinor

In [40]:
# Location of each series in the Norsk Petroleum net cash flow file, Excel row/column numbers
# years run DOWN a column here (unlike IFAC where years run across a row)
# key: (sheet, year column, first data row, unit, scale, {data column: feature_name})
# values are NOK billion in constant 2027 prices, paid taxes adjusted for repayments
noroil_tabs_dict = {
    "net_cash_flow": ("Fig-data", 2, 24, "NOK Billion (2027 prices)", 1,
                {4: "noroil_ord_tax",         # 22% ordinary company tax - comparable to Irish CT
                 5: "noroil_spec_tax",          # 71.8% special tax on oil company, 
                 6: "noroil_ord_plus_special_tax",        # ordinary + special tax combined to get a total tax value
                 11: "noroil_sdfi_net_cash_flow",  # state's income from share of ownership of the oil fields themselves,owned by a company called Petoro (gov co.)
                 12: "noroil_equinor_dividend",    # dividend from states 67%  shareholding in equinor
                 13: "noroil_state_net_cash_flow"}),  # total,i.e. ord +spec +sdfi royalty+equinor div+env tax+royalties
}

In [57]:

# Function to extract columns from a sheet where years run down a column, into the same 7 column long format
def col_extr_conv_fn(xl, org, file_tag, tab_key, sheet, year_col, first_row, unit, scale, cols, last_actual=2025):
    # whole sheet as a grid, no header
    raw = pd.read_excel(xl, sheet_name=sheet, header=None)
    # years from the year column, from the first data row down, text/blanks -> nan
    # had yo modify for uk data, make eveyr cel str text, and find all 4 digits, take the first one and ocnvert to a number
    # was droppign values as they werent' numbers in UK data for 
    years = pd.to_numeric(raw.iloc[first_row - 1:, year_col - 1].astype(str).str.findall(r"(?:19|20)\d{2}").str[0], errors="coerce")
    out = []
    # loop through the chosen columns, one feature per column
    for c, feature in cols.items():
        vals = pd.to_numeric(raw.iloc[first_row - 1:, c - 1], errors="coerce") * scale
        df = pd.DataFrame({"year": years.values, "value": vals.values}).dropna()
        df["year"] = df["year"].astype(int)
        df["feature_name"] = feature
        df["unit"] = unit
        df["orig_org_source"] = org
        df["orig_file_source"] = f"{file_tag}_{tab_key}"
        df["is_forecast"] = df["year"] > last_actual
        out.append(df)
    return pd.concat(out, ignore_index=True)[
        ["year", "feature_name", "value", "unit", "orig_org_source", "orig_file_source", "is_forecast"]]


In [42]:
# Extract norway xlsx file into data_sources
#  xlsx wa saken from here > https://www.norskpetroleum.no/en/economy/governments-revenues
# some other sources but thsi seemed the most relevant
noroil_file = data_location / "30_Statens-netto-kontantstrom_1971-2025_07102026.xlsx"
for tab_key, (sheet, year_col, first_row, unit, scale, cols) in noroil_tabs_dict.items():
    df_noroil_1 = col_extr_conv_fn(noroil_file, "NORSKPETROLEUM", "noroil", tab_key, sheet, year_col, first_row, unit, scale, cols)
    key = f"noroil_{tab_key}"
    data_sources[key] = df_noroil_1
    print(f"{key} : {df_noroil_1.shape} <-{noroil_file.name} [{sheet}]")

df_noroil_1 = data_sources["noroil_net_cash_flow"]

noroil_net_cash_flow : (330, 7) <-30_Statens-netto-kontantstrom_1971-2025_07102026.xlsx [Fig-data]


In [43]:
# Convert Used ecb.com average NOK to euro exchange rate of  1 euro = 11.7143 NOK. Norwegian krone (NOK)
nok_to_euro_exch_rate = 11.7

df_noroil_2 = df_noroil_1.copy()

# wil create a new feature name for nor oil euros, keeping NOK values if needed.
df_noroil_2["feature_name"] = df_noroil_2["feature_name"] + "_eur"
df_noroil_2["value"] = df_noroil_2["value"] * 1000 / nok_to_euro_exch_rate   # NOK bn -> NOK m -> EUR m
df_noroil_2["unit"] = "Euro Million (constant prices)"
df_noroil_2["orig_file_source"] = "calculated"

df_noroil_3 = pd.concat([df_noroil_1, df_noroil_2], ignore_index=True)
print(df_noroil_3.shape)   # expect (660, 7)


(660, 7)


In [44]:
print(df_noroil_3.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 660 entries, 0 to 659
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              660 non-null    int64  
 1   feature_name      660 non-null    object 
 2   value             660 non-null    float64
 3   unit              660 non-null    object 
 4   orig_org_source   660 non-null    object 
 5   orig_file_source  660 non-null    object 
 6   is_forecast       660 non-null    bool   
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 31.7+ KB
None


In [45]:
print(df_noroil_3.describe())

              year          value
count   660.000000     660.000000
mean   1998.000000    4874.894404
std      15.886548   11639.092562
min    1971.000000   -3181.396250
25%    1984.000000      20.027675
50%    1998.000000     193.710225
75%    2012.000000    3559.430234
max    2025.000000  128042.115295


In [46]:
# quick look at data to see if its clean, no duplicates, null values unusual characters etc
# convert it to d first so can easily run some checks over it
# df=pd.DataFrame(processed_per_file_data)
# view various dataframe details
# checking for null values and specific unknown values
print ("Rows     : " , df_noroil_3.shape[0])
print ("Columns  : " , df_noroil_3.shape[1])
print('------')
print ("\nFeatures : \n" , df_noroil_3.columns.tolist())
print('------')
print ("\nMissing values :  ", df_noroil_3.isnull().sum().values.sum())
print('------')
print ("\nMissing values per column:  ", df_noroil_3.isnull().sum())
print('------')
print('\nnum of dups in df   =   {}'.format(df_noroil_3.duplicated().sum()))
print('------')
print ("\nUnique values :  \n", df_noroil_3.nunique())
print('------')
print('\n null values = ', df_noroil_3.isnull().values.any())
print('------')
df_cso_missing = (df_noroil_3=='?').sum()
print('\"  ?   \" values present =  \n',df_cso_missing)
print('------')
df_cso_unknown = (df_noroil_3=='unknown').sum()
print('\"   unknown   \" values present =  \n',df_cso_unknown)
print('------')
df_cso_na = df_noroil_3.isin(['N/A', 'N\\A', 'NA', 'n/a', 'n\\A', 'na', 'N_A']).sum()
print('    NA     values present = \n', df_cso_na)
print('------')
df_cso_none = df_noroil_3.isin(['None', 'none']).sum()
print('    none    values present = \n', df_cso_none)
print('------')
# regex https://stackoverflow.com/questions/2124010/grep-regex-to-match-non-ascii-characters
# count of rows per column containing non-ascii characters
rogue_ascii_df_cso = df_noroil_3.apply(lambda s: s.astype(str).str.contains(r"[^\x00-\x7F]", regex=True, na=False).sum())
print("    non ascii chars   values present:\n", rogue_ascii_df_cso)

Rows     :  660
Columns  :  7
------

Features : 
 ['year', 'feature_name', 'value', 'unit', 'orig_org_source', 'orig_file_source', 'is_forecast']
------

Missing values :   0
------

Missing values per column:   year                0
feature_name        0
value               0
unit                0
orig_org_source     0
orig_file_source    0
is_forecast         0
dtype: int64
------

num of dups in df   =   0
------

Unique values :  
 year                 55
feature_name         12
value               579
unit                  2
orig_org_source       1
orig_file_source      2
is_forecast           1
dtype: int64
------

 null values =  False
------
"  ?   " values present =  
 year                0
feature_name        0
value               0
unit                0
orig_org_source     0
orig_file_source    0
is_forecast         0
dtype: int64
------
"   unknown   " values present =  
 year                0
feature_name        0
value               0
unit                0
orig_org_sourc

In [47]:
# checkign output
df_noroil_final = df_noroil_3
print(f" {(df_noroil_final.shape)}  -- expecting 660 rows and 7 columns, within which are 12 features")
print("==========")
print(" ----features names ---")
print(f" {df_noroil_final["feature_name"].unique()}")

 (660, 7)  -- expecting 660 rows and 7 columns, within which are 12 features
 ----features names ---
 ['noroil_ord_tax' 'noroil_spec_tax' 'noroil_ord_plus_special_tax'
 'noroil_sdfi_net_cash_flow' 'noroil_equinor_dividend'
 'noroil_state_net_cash_flow' 'noroil_ord_tax_eur' 'noroil_spec_tax_eur'
 'noroil_ord_plus_special_tax_eur' 'noroil_sdfi_net_cash_flow_eur'
 'noroil_equinor_dividend_eur' 'noroil_state_net_cash_flow_eur']


In [48]:
# combine nor oil df into CSO_IFAC df

df_cso_ifac_noroil = pd.concat([df_cso_ifac, df_noroil_final], ignore_index=True)

In [49]:
print(f" df_cso_ifac:  {df_cso_ifac.shape}")
print(f" df_noroil_final: {df_noroil_final.shape}")
print(f"=================================")
print(f" Combined  df_cso_ifac_noroil shape = {df_cso_ifac_noroil.shape} - expecting 1364 rows (cso=519 + ifac=185 + ifac=660) and 7 features (both =7)")

 df_cso_ifac:  (704, 7)
 df_noroil_final: (660, 7)
 Combined  df_cso_ifac_noroil shape = (1364, 7) - expecting 1364 rows (cso=519 + ifac=185 + ifac=660) and 7 features (both =7)


In [50]:
# Export into csv we cna import import into Tableau then

# export to csv so cna load into Tableau and do explorattory data visualisations
# define the path and create the folder if it doesn' exist
print(f" export location = {export_location}" )
export_filename_cso_ifac_noroil = ("df_cso_ifac_noroil.csv")
export_location.mkdir(exist_ok=True)

df_cso_ifac_noroil.to_csv(export_location/export_filename_cso_ifac_noroil , index=False)
print(f" saved file : {(export_location/export_filename_cso_ifac_noroil).resolve()}")


 export location = exported
 saved file : E:\GG\Msc\SPEC9995_data_visualisation_marisa llorens-salvador_tues\github\spec9995_data_vis_marissa_tues\assignment 1\working\exported\df_cso_ifac_noroil.csv


# UK North Sea Oil

In [61]:
# mapping

# Location of each series in HMRC Government revenues from oil and gas production, October 2026, Table 2
# years run DOWN column A as UK fiscal years ("2024 to 2025") - mapped to start year (Apr-Dec = 9 of 12 months)
# key: (sheet, year column, first data row, unit, scale, {data column: feature_name})
# values in GBP million, nominal, net of repayments
ukoil_tabs_dict = {
    "table_2": ("Table_2", 1, 7, "GBP Million", 1,
                {2: "ukoil_total_net_revenue",   # total net revenue from oil and gas production - UK tax take
                 4: "ukoil_ring_fncd_ct",                # ring fence corporation tax (30%) - closest to Irish CT
                 5: "ukoil_supp_charge",         # supplementary charge on ring fence profits (since 2002)
                 6: "ukoil_royalty",             # royalty on production (abolished 2003)
                 8: "ukoil_pet_rev_tax",                 # petroleum revenue tax (zero-rated from 2016, now net repayments)
                 9: "ukoil_nrg_prfts_levy"}),               # energy profits levy - windfall tax since 2022
}

In [62]:
# Extract uk ods file into data_sources
#  ods formatted file was taken from here > https://www.gov.uk/government/statistics/government-revenues-from-uk-oil-and-gas-production--2
# some other sources but thsi seemed the most relevant
ukoil_file = data_location / "Government_revenues_from_oil_and_gas_production_October_2026_-_data_tables.ods"
for tab_key, (sheet, year_col, first_row, unit, scale, cols) in ukoil_tabs_dict.items():
    df_ukoil_1 = col_extr_conv_fn(ukoil_file, "HMRC", "ukoil", tab_key, sheet, year_col, first_row, unit, scale, cols)
    key = f"ukoil_{tab_key}"
    data_sources[key] = df_ukoil_1
    print(f"{key} : {df_ukoil_1.shape} <-{ukoil_file.name} [{sheet}]")

df_noroil_1 = data_sources["noroil_net_cash_flow"]

ukoil_table_2 : (226, 7) <-Government_revenues_from_oil_and_gas_production_October_2026_-_data_tables.ods [Table_2]


In [63]:
print(df_ukoil_1["orig_org_source"].unique())   # expect ['HMRC']


['HMRC']


In [64]:
# Convert GBP to Euro using ECB 2025 average exchange rate, 1 euro = X GBP
# we will follow the same appraoch for sterling, i.e.  asingle fixed rate used for consistency with Norway, and as no EUR/GBP rate exists before 1999
# amoutns already in millinos so no conversion need as was the case with norway in billions
gbp_to_euro_exch_rate = 0.85   # 0.8569 - average from 01/01/2025 to 31/12/2025

df_ukoil_2 = df_ukoil_1.copy()

# create new feature names for uk oil euros, keeping GBP values as well
df_ukoil_3 = df_ukoil_2.copy()
df_ukoil_3["feature_name"] = df_ukoil_3["feature_name"] + "_eur"
df_ukoil_3["value"] = df_ukoil_3["value"] / gbp_to_euro_exch_rate   # GBP m -> EUR m
df_ukoil_3["unit"] = "Euro Million"
df_ukoil_3["orig_file_source"] = "calculated"

df_ukoil_4 = pd.concat([df_ukoil_2, df_ukoil_3], ignore_index=True)

df_ukoil_final = df_ukoil_4
print(f" {df_ukoil_final.shape}  -- expecting 452 rows and 7 columns")
print(f" {df_ukoil_final['feature_name'].unique()}")

 (452, 7)  -- expecting 452 rows and 7 columns
 ['ukoil_total_net_revenue' 'ukoil_ring_fncd_ct' 'ukoil_supp_charge'
 'ukoil_royalty' 'ukoil_pet_rev_tax' 'ukoil_nrg_prfts_levy'
 'ukoil_total_net_revenue_eur' 'ukoil_ring_fncd_ct_eur'
 'ukoil_supp_charge_eur' 'ukoil_royalty_eur' 'ukoil_pet_rev_tax_eur'
 'ukoil_nrg_prfts_levy_eur']


In [65]:
# combine UK oil df into CSO_IFAC_noroil  df

df_cso_ifac_noroil_ukoil = pd.concat([df_cso_ifac_noroil, df_ukoil_final], ignore_index=True)

In [68]:
print(f" df_cso_ifac_noroil:  {df_cso_ifac_noroil.shape}")
print(f" df_ukoil_final: {df_ukoil_final.shape}")
print(f"=================================")
print(f" Combined  df_cso_ifac_noroil_ukoil shape = {df_cso_ifac_noroil_ukoil.shape} - expecting 1816 rows (cso=519 + ifac=185 + ifac=660 + uk=452) and 7 features (both =7)")

 df_cso_ifac_noroil:  (1364, 7)
 df_ukoil_final: (452, 7)
 Combined  df_cso_ifac_noroil_ukoil shape = (1816, 7) - expecting 1816 rows (cso=519 + ifac=185 + ifac=660 + uk=452) and 7 features (both =7)


In [70]:
# Export into csv we cna import import into Tableau then

# export to csv so cna load into Tableau and do explorattory data visualisations
# define the path and create the folder if it doesn' exist
print(f" export location = {export_location}" )
export_filename_cso_ifac_noroil_ukoil = ("df_cso_ifac_noroil_ukoil.csv")
export_location.mkdir(exist_ok=True)

df_cso_ifac_noroil_ukoil.to_csv(export_location/export_filename_cso_ifac_noroil_ukoil , index=False)
print(f" saved file : {(export_location/export_filename_cso_ifac_noroil_ukoil).resolve()}")

 export location = exported
 saved file : E:\GG\Msc\SPEC9995_data_visualisation_marisa llorens-salvador_tues\github\spec9995_data_vis_marissa_tues\assignment 1\working\exported\df_cso_ifac_noroil_ukoil.csv


# Dutch Gas 

In [ ]:
# mapping

# Location of each series in HMRC Government revenues from oil and gas production, October 2026, Table 2
# years run DOWN column A as UK fiscal years ("2024 to 2025") - mapped to start year (Apr-Dec = 9 of 12 months)
# key: (sheet, year column, first data row, unit, scale, {data column: feature_name})
# values in GBP million, nominal, net of repayments
nlgas_tabs_dict = {
   "nominal_1969_2000": ("Totaaloverzicht lopende prijzen", 1, 6, "Euro Million", 1,
                {2: "nlgas_total_revenue"}),          # total gas revenue, current (nominal) prices
    "nominal_2001_2021": ("Totaaloverzicht lopende prijzen", 4, 38, "Euro Million", 1,
                {5: "nlgas_total_revenue"}),
    "real_1969_2000":    ("Totaaloverzicht prijzen 2021", 1, 6, "Euro Million (2021 prices)", 1,
                {2: "nlgas_total_revenue_real"}),     # same series in constant 2021 prices
    "real_2001_2021":    ("Totaaloverzicht prijzen 2021", 4, 38, "Euro Million (2021 prices)", 1,
                {5: "nlgas_total_revenue_real"}),
}